# MP2

In [1]:
from pyscf import gto, scf, mp, ao2mo
import numpy as np
from pathlib import Path
# from py_mods.src.SCF.CSUHF import CS_UHF_ContextClass, CS_UHF
from py_mods.src.SCF.CSRHF import CS_RHF, CS_RHF_ContextClass
from py_mods.src.SCF.plot_utilities import plot_map
from Dev.CSMP2_dev import CS_MP2

In [2]:
# pyscf data
mol = gto.M(atom = 'He 0 0 0', spin=0, charge=0, basis='aug-cc-pvqz')
n_elec = sum(mol.nelec)

kin = mol.intor('int1e_kin')
vnuc = mol.intor('int1e_nuc')
overlap = mol.intor('int1e_ovlp')
eri = mol.intor('int2e')

mf = scf.RHF(mol) 

e_He = mf.kernel()
e_elec = mf.energy_elec()

mymp = mp.RMP2(mf).run() # this is UMP2
eris_mo = ao2mo.kernel(mol, mf.mo_coeff, aosym='1')

converged SCF energy = -2.86152199563245
E(RMP2) = -2.89724612518338  E_corr = -0.0357241295509312
E(SCS-RMP2) = -2.90439095109357  E_corr = -0.0428689554611174


In [3]:

# implementation and calculation
Li_context = CS_RHF_ContextClass(overlap, kin, vnuc, eri, n_electrons=n_elec, theta=0.0, verbose=False, threshold=1E-7)
RHF_res = CS_RHF(Li_context)
print(f'\nSCF energy: {RHF_res.E_RHF.real} (converged: {RHF_res.converged})')
print(f'SCF pyscf: {e_He}')
print(f'Difference: {RHF_res.E_RHF.real - e_He} \n')


SCF energy: -2.861521995632467 (converged: True)
SCF pyscf: -2.8615219956324496
Difference: -1.7319479184152442e-14 



In [4]:

mp_results = CS_MP2(RHF_res)

print(f'\n\nMP2 calc: {mp_results.E_MP2}, E_corr = {mp_results.E_corr}')
print(f'MP2 pyscf: {mymp.e_tot}, E_corr = {mymp.e_corr}')
print(f'Differences: {mp_results.E_MP2 - mymp.e_tot}, E_corr = {mp_results.E_corr - mymp.e_corr}\n')


Number of occupied orbitals: 1
Number of virtual orbitals: 45
Number of total orbitals: 46


MP2 calc: (-2.9001969607573814+0j), E_corr = -0.03867496512491447
MP2 pyscf: -2.897246125183381, E_corr = -0.03572412955093121
Differences: (-0.0029508355740004433+0j), E_corr = -0.0029508355739832626



In [5]:
er_py = eris_mo
er_im = mp_results.eris_mo

In [6]:
np.allclose(er_im, er_py.reshape(er_im.shape))

False

In [7]:
# to see later, the exact use of slices in this https://pycrawfordprogproj.readthedocs.io/en/latest/Project_04/Project_04.html